# 🏷️ Fine-Tuning DistilBERT for POS Tagging & Chunking
## Data Science Internship – February 2026 | Task 5: NLP

---

## 📌 Objective
Fine-tune a pre-trained **DistilBERT** transformer model to perform:
- **POS Tagging** → Assign grammatical tags to each word (Noun, Verb, Adjective, etc.)
- **Chunking** → Group words into meaningful phrases (NP, VP, PP, etc.)

---

## 📖 Key Concepts

### 🔹 What is POS Tagging?
**Part-of-Speech (POS) Tagging** assigns a grammatical label to each word in a sentence.

| Word | POS Tag | Meaning |
|------|---------|---------|
| John | NNP | Proper Noun |
| runs | VBZ | Verb (3rd person singular) |
| fast | RB | Adverb |

### 🔹 What is Chunking?
**Chunking** groups words into meaningful phrases based on POS tags.

| Chunk | Meaning | Example |
|-------|---------|---------|
| NP | Noun Phrase | "The big dog" |
| VP | Verb Phrase | "is running" |
| PP | Prepositional Phrase | "in the park" |

### 🔹 What is Token Classification?
Unlike sentence classification, **Token Classification** assigns a label to **each token** in the sequence.

---

## 🔄 Pipeline Overview

In [2]:
# ─────────────────────────────────────────────────
# VERIFY ENVIRONMENT
# ─────────────────────────────────────────────────

!pip install seqeval
!pip install evaluate

import transformers
import datasets
import seqeval
import evaluate
import accelerate
import torch

print("=" * 50)
print("✅ transformers :", transformers.__version__)
print("✅ datasets     :", datasets.__version__)
print("✅ evaluate     :", evaluate.__version__)
print("✅ accelerate   :", accelerate.__version__)
print("✅ torch        :", torch.__version__)
print("✅ seqeval      : loaded successfully")
print("=" * 50)

print(f"🖥️ GPU Available : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"🖥️ GPU Name     : {torch.cuda.get_device_name(0)}")
print("=" * 50)

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 3.2 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for seqeval: filename=seqeval-1.2.2-py3-none-any.whl size=16162 sha256=582b77c0dbdf96be4ca17f40cfefa3f6538fca6c32d62574354254ea5bdd2376
  Stored in directory: /root/.cache/pip/wheels/5f/b8/73/0b2c1a76b701a677653dd79ece07cfabd7457989dbfbdcd8d7
Successfully built seqeval
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 4.9 MB/s eta 0:00:00
✅ transformers : 5.0.0
✅ datasets     : 4.0.0
✅ evaluate     : 0.4.6
✅ accelerate   : 1.13.0
✅ torch        : 2.10.0+cu128
✅ seqeval      : loaded successfully
🖥️ GPU Available : True
🖥️ GPU Name     : Tesla T4


## 🔧 Step 1: Verify & Import Libraries

### Theory: Why These Libraries?

| Library | Purpose |
|---------|---------|
| `transformers` | DistilBERT model & tokenizer — Hugging Face |
| `datasets` | Load and process CoNLL-2003 dataset |
| `seqeval` | Sequence-level evaluation metrics for NLP |
| `evaluate` | Hugging Face evaluation framework |
| `accelerate` | Speeds up PyTorch training on GPU |
| `torch` | PyTorch deep learning framework |

### Why DistilBERT?
- **40% smaller** than BERT-base
- **60% faster** inference speed
- Retains **97% of BERT's performance**
- Perfect for token classification tasks

In [3]:
import os
import numpy as np
import pandas as pd
import torch
from google.colab import files
from transformers import (
    AutoTokenizer,
    AutoModelForTokenClassification,
    TrainingArguments,
    Trainer,
    DataCollatorForTokenClassification
)
from datasets import Dataset
import evaluate
from seqeval.metrics import (
    classification_report,
    precision_score,
    recall_score,
    f1_score
)

print("✅ All imports successful!")
print(f"🖥️  GPU : {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NOT FOUND'}")

✅ All imports successful!
🖥️  GPU : Tesla T4


## 📥 Step 2: Load CoNLL-2003 Dataset

### Theory: CoNLL-2003 Dataset

| Property | Details |
|----------|---------|
| Source | Conference on Natural Language Learning 2003 |
| Task | POS Tagging + Chunking + NER |
| Train Size | ~14,000 sentences |
| Val Size | ~3,400 sentences |
| Test Size | ~3,600 sentences |

### Dataset Format:

Each line contains 4 columns:

# WORD | POS-TAG| CHUNK-TAG | NER-TAG

## EU      | NNP   | B-NP    | B-ORG
## rejects | VBZ   | B-VP    | O
## German  | JJ    |B-NP     | B-MISC

### BIO Tagging Scheme:
- **B-** → Beginning of a chunk
- **I-** → Inside a chunk
- **O**  → Outside any chunk

### Files Required:
| File | Split |
|------|-------|
| `eng.train` | Training data |
| `eng.testa` | Validation data |
| `eng.tstb` | Test data |

In [4]:
print("⬆️  Upload your 3 files: eng.train, eng.testa, eng.tstb")
uploaded = files.upload()
print(f"\n✅ Files uploaded: {list(uploaded.keys())}")

⬆️  Upload your 3 files: eng.train, eng.testa, eng.tstb


Saving eng.testa to eng.testa
Saving eng.testb to eng.testb
Saving eng.train to eng.train

✅ Files uploaded: ['eng.testa', 'eng.testb', 'eng.train']


## 🧹 Step 3: Data Preprocessing

### Theory: Parsing CoNLL Format

Raw CoNLL files are in a specific text format:
- Each **word** is on a separate line with its tags
- **Blank lines** separate sentences
- Lines starting with `-DOCSTART-` are headers — ignored

### Parsing Steps:
1. Read file line by line
2. Skip `-DOCSTART-` headers
3. On blank line → save current sentence, start new one
4. Extract: `word`, `POS tag`, `Chunk tag`

### Example:
```
Input line  : "EU  NNP  B-NP  B-ORG"
Extracted   : word="EU", pos="NNP", chunk="B-NP"
```

In [5]:
def parse_conll_file(filepath):
    """
    Reads CoNLL-2003 file.
    Each line: WORD  POS-TAG  CHUNK-TAG  NER-TAG
    Blank lines = sentence boundary
    """
    sentences = []
    words, pos_tags, chunk_tags = [], [], []

    with open(filepath, 'r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()

            if line.startswith('-DOCSTART-'):
                continue

            if line == '':
                if words:
                    sentences.append({
                        'tokens'     : words,
                        'pos_tags'   : pos_tags,
                        'chunk_tags' : chunk_tags
                    })
                    words, pos_tags, chunk_tags = [], [], []
            else:
                parts = line.split()
                if len(parts) >= 3:
                    words.append(parts[0])
                    pos_tags.append(parts[1])
                    chunk_tags.append(parts[2])

    # Catch last sentence
    if words:
        sentences.append({
            'tokens'     : words,
            'pos_tags'   : pos_tags,
            'chunk_tags' : chunk_tags
        })

    return sentences


# ── Auto-detect test filename ──
test_filename = 'eng.tstb' if os.path.exists('eng.tstb') else 'eng.testb'

train_data = parse_conll_file('eng.train')
val_data   = parse_conll_file('eng.testa')
test_data  = parse_conll_file(test_filename)

print(f"✅ Train      : {len(train_data):,} sentences")
print(f"✅ Validation : {len(val_data):,} sentences")
print(f"✅ Test       : {len(test_data):,} sentences")

# ── Preview sample ──
s = train_data[2]
print(f"\n📄 Sample Preview:")
print(f"  {'Word':<18} {'POS':<10} {'Chunk':<12}")
print(f"  {'─'*18} {'─'*10} {'─'*12}")
for w, p, c in zip(s['tokens'][:8], s['pos_tags'][:8], s['chunk_tags'][:8]):
    print(f"  {w:<18} {p:<10} {c:<12}")

✅ Train      : 14,041 sentences
✅ Validation : 3,250 sentences
✅ Test       : 3,453 sentences

📄 Sample Preview:
  Word               POS        Chunk       
  ────────────────── ────────── ────────────
  BRUSSELS           NNP        B-NP        
  1996-08-22         CD         I-NP        


## 🏷️ Step 4: Label Setup & Task Selection

### Theory: Label Mappings

For token classification, we need to convert **string labels** to **integer IDs**:

| Label | ID |
|-------|----|
| B-NP  | 0  |
| B-VP  | 1  |
| I-NP  | 2  |
| O     | 3  |
| ...   | ...|

### Two Tasks Available:
| Task | Label Key | Example Labels |
|------|-----------|---------------|
| `chunk` | `chunk_tags` | B-NP, I-NP, B-VP, O |
| `pos` | `pos_tags` | NN, VB, JJ, RB |

### id2label & label2id:
- `label2id` → converts label string to integer (for training)
- `id2label` → converts integer back to label string (for inference)

In [6]:
# ── Collect all unique labels from training data ──
all_chunk_labels = sorted(set(t for s in train_data for t in s['chunk_tags']))
all_pos_labels   = sorted(set(t for s in train_data for t in s['pos_tags']))

# ── Choose task: 'chunk' or 'pos' ──
TASK = 'chunk'

if TASK == 'chunk':
    all_labels = all_chunk_labels
    label_key  = 'chunk_tags'
else:
    all_labels = all_pos_labels
    label_key  = 'pos_tags'

label2id   = {label: idx for idx, label in enumerate(all_labels)}
id2label   = {idx: label for label, idx in label2id.items()}
NUM_LABELS = len(label2id)

print(f"✅ Task       : {TASK.upper()} Tagging")
print(f"✅ Labels     : {NUM_LABELS}")
print(f"✅ Label list : {all_labels}")

✅ Task       : CHUNK Tagging
✅ Labels     : 20
✅ Label list : ['B-ADJP', 'B-ADVP', 'B-CONJP', 'B-INTJ', 'B-LST', 'B-NP', 'B-PP', 'B-PRT', 'B-SBAR', 'B-VP', 'I-ADJP', 'I-ADVP', 'I-CONJP', 'I-INTJ', 'I-LST', 'I-NP', 'I-PP', 'I-SBAR', 'I-VP', 'O']


In [7]:
MODEL_NAME = "distilbert-base-uncased"
tokenizer  = AutoTokenizer.from_pretrained(MODEL_NAME)
print(f"✅ Tokenizer loaded: {MODEL_NAME}")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

✅ Tokenizer loaded: distilbert-base-uncased


## 🔤 Step 5: Tokenization & Label Alignment

### Theory: The Subword Problem

DistilBERT uses **WordPiece tokenization** which splits words into subwords:

- Word    : "playing"
- Tokens  : ["play", "##ing"]

This creates a mismatch — we have **1 label per word** but **2+ tokens per word**.

### Solution: Label Alignment

| Token   | Word ID | Label |
|---------|---------|-------|
| [CLS]   | None    | -100  |
| play    | 0       | B-VP  |
| ##ing   | 0       | -100  |
| [SEP]   | None    | -100  |

### Rules:
- First subword → gets the **real label**
- Remaining subwords → get **-100** (ignored in loss)
- `[CLS]` and `[SEP]` → get **-100**

### Why -100?
PyTorch's CrossEntropyLoss **automatically ignores** positions with label `-100`.

In [8]:
# ─────────────────────────────────────────────────
# CELL 7 — FIXED: Tokenize & Align Labels
# Fix: Build label2id from ALL splits (train+val+test)
# so no unknown label causes a KeyError
# ─────────────────────────────────────────────────

# ── Rebuild label mappings from ALL data ──
all_data = train_data + val_data + test_data

all_chunk_labels = sorted(set(t for s in all_data for t in s['chunk_tags']))
all_pos_labels   = sorted(set(t for s in all_data for t in s['pos_tags']))

# Apply to chosen task
if TASK == 'chunk':
    all_labels = all_chunk_labels
    label_key  = 'chunk_tags'
else:
    all_labels = all_pos_labels
    label_key  = 'pos_tags'

label2id   = {label: idx for idx, label in enumerate(all_labels)}
id2label   = {idx: label for label, idx in label2id.items()}
NUM_LABELS = len(label2id)

print(f"✅ Labels rebuilt from ALL splits : {NUM_LABELS} labels")
print(f"✅ Labels : {all_labels}")


# ── Tokenize & Align ──
def tokenize_and_align_labels(examples):
    tokenized_inputs = tokenizer(
        examples['tokens'],
        truncation          = True,
        is_split_into_words = True,
        max_length          = 128,
        padding             = False
    )

    all_labels_out = []
    for i, labels in enumerate(examples[label_key]):
        word_ids      = tokenized_inputs.word_ids(batch_index=i)
        previous_word = None
        label_ids     = []

        for word_idx in word_ids:
            if word_idx is None:
                label_ids.append(-100)
            elif word_idx != previous_word:
                # ── Safe lookup with fallback ──
                tag = labels[word_idx]
                label_ids.append(label2id.get(tag, -100))  # -100 if unseen tag
            else:
                label_ids.append(-100)
            previous_word = word_idx

        all_labels_out.append(label_ids)

    tokenized_inputs['labels'] = all_labels_out
    return tokenized_inputs


# ── Convert to HuggingFace Dataset ──
def to_hf_dataset(data):
    return Dataset.from_dict({
        'tokens'     : [s['tokens']     for s in data],
        'pos_tags'   : [s['pos_tags']   for s in data],
        'chunk_tags' : [s['chunk_tags'] for s in data],
    })

remove_cols = ['tokens', 'pos_tags', 'chunk_tags']

train_tokenized = to_hf_dataset(train_data).map(
    tokenize_and_align_labels, batched=True, remove_columns=remove_cols)
val_tokenized   = to_hf_dataset(val_data).map(
    tokenize_and_align_labels, batched=True, remove_columns=remove_cols)
test_tokenized  = to_hf_dataset(test_data).map(
    tokenize_and_align_labels, batched=True, remove_columns=remove_cols)

print(f"\n✅ Train tokenized : {len(train_tokenized)}")
print(f"✅ Val   tokenized : {len(val_tokenized)}")
print(f"✅ Test  tokenized : {len(test_tokenized)}")

# ── Verify sample ──
sample = train_tokenized[0]
print(f"\n📌 input_ids      : {sample['input_ids'][:8]}")
print(f"📌 attention_mask : {sample['attention_mask'][:8]}")
print(f"📌 labels         : {sample['labels'][:8]}")

✅ Labels rebuilt from ALL splits : 21 labels
✅ Labels : ['B-ADJP', 'B-ADVP', 'B-CONJP', 'B-INTJ', 'B-LST', 'B-NP', 'B-PP', 'B-PRT', 'B-SBAR', 'B-VP', 'I-ADJP', 'I-ADVP', 'I-CONJP', 'I-INTJ', 'I-LST', 'I-NP', 'I-PP', 'I-PRT', 'I-SBAR', 'I-VP', 'O']


Map:   0%|          | 0/14041 [00:00<?, ? examples/s]

Map:   0%|          | 0/3250 [00:00<?, ? examples/s]

Map:   0%|          | 0/3453 [00:00<?, ? examples/s]


✅ Train tokenized : 14041
✅ Val   tokenized : 3250
✅ Test  tokenized : 3453

📌 input_ids      : [101, 7327, 19164, 2446, 2655, 2000, 17757, 2329]
📌 attention_mask : [1, 1, 1, 1, 1, 1, 1, 1]
📌 labels         : [-100, 5, 9, 5, 15, 9, 19, 5]


## 🏗️ Step 6: Model Setup

### Theory: AutoModelForTokenClassification

| Component | Details |
|-----------|---------|
| Base Model | DistilBERT-base-uncased |
| Layers | 6 Transformer layers |
| Hidden Size | 768 |
| Parameters | ~66 Million |
| Classification Head | Linear(768 → num_labels) |

### How it Works:

### Input Tokens
## ↓
### DistilBERT Encoder (6 layers)
## ↓
### Hidden States for EACH token → shape: (batch, seq_len, 768)
## ↓
### Dropout Layer
## ↓
### Linear Layer (768 → num_labels)
## ↓
### Logits per token → argmax → Predicted Label

### DistilBERT vs BERT:


| Property | BERT-base | DistilBERT |
|----------|-----------|------------|
| Layers | 12 | 6 |
| Parameters | 110M | 66M |
| Speed | 1x | 1.6x faster |
| Size | 440MB | 265MB |
| Accuracy | 100% | ~97% of BERT |

In [9]:
model = AutoModelForTokenClassification.from_pretrained(
    MODEL_NAME,
    num_labels              = NUM_LABELS,
    id2label                = id2label,
    label2id                = label2id,
    ignore_mismatched_sizes = True
)

print(f"✅ Model loaded     : {MODEL_NAME}")
print(f"✅ Total parameters : {model.num_parameters():,}")
print(f"✅ Output labels    : {NUM_LABELS}")

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForTokenClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


✅ Model loaded     : distilbert-base-uncased
✅ Total parameters : 66,379,029
✅ Output labels    : 21


## 🏋️ Step 7: Training Setup & Fine-Tuning

### Theory: Hugging Face Trainer API

The `Trainer` class handles:
- Training loop automatically
- Gradient updates
- Evaluation after each epoch
- Saving best model

### Hyperparameters:

| Parameter | Value | Reason |
|-----------|-------|--------|
| Learning Rate | 2e-5 | Standard for transformer fine-tuning |
| Epochs | 3 | Sufficient for token classification |
| Train Batch Size | 16 | Memory efficient |
| Eval Batch Size | 32 | Faster evaluation |
| Weight Decay | 0.01 | Regularization to prevent overfitting |
| fp16 | True | Mixed precision — faster GPU training |

### Data Collator:
`DataCollatorForTokenClassification` handles **dynamic padding** — pads each batch to the longest sequence in that batch (more efficient than padding to MAX_LEN always).

In [10]:
data_collator  = DataCollatorForTokenClassification(tokenizer=tokenizer)
seqeval_metric = evaluate.load("seqeval")

def compute_metrics(p):
    logits, labels = p
    predictions    = np.argmax(logits, axis=2)

    true_predictions = [
        [id2label[int(pred)] for pred, lbl in zip(row_p, row_l) if int(lbl) != -100]
        for row_p, row_l in zip(predictions, labels)
    ]
    true_labels = [
        [id2label[int(lbl)] for pred, lbl in zip(row_p, row_l) if int(lbl) != -100]
        for row_p, row_l in zip(predictions, labels)
    ]

    results = seqeval_metric.compute(
        predictions = true_predictions,
        references  = true_labels
    )
    return {
        'precision' : round(results['overall_precision'], 4),
        'recall'    : round(results['overall_recall'],    4),
        'f1'        : round(results['overall_f1'],        4),
    }

print("✅ Data collator and metrics ready!")

✅ Data collator and metrics ready!


In [11]:

training_args = TrainingArguments(
    output_dir                  = './results',
    num_train_epochs            = 3,
    per_device_train_batch_size = 16,
    per_device_eval_batch_size  = 32,
    learning_rate               = 2e-5,
    weight_decay                = 0.01,
    eval_strategy               = 'epoch',
    save_strategy               = 'epoch',
    load_best_model_at_end      = True,
    metric_for_best_model       = 'f1',
    logging_steps               = 200,
    report_to                   = 'none',
    fp16                        = True,
)

trainer = Trainer(
    model            = model,
    args             = training_args,
    train_dataset    = train_tokenized,
    eval_dataset     = val_tokenized,
    processing_class = tokenizer,        # ✅ fixed (was tokenizer=)
    data_collator    = data_collator,
    compute_metrics  = compute_metrics,
)

print("🚀 Training started...")
train_result = trainer.train()
print(f"\n✅ Training complete!")
print(f"   Loss  : {train_result.training_loss:.4f}")
print(f"   Steps : {train_result.global_step}")

🚀 Training started...


Epoch,Training Loss,Validation Loss,Precision,Recall,F1
1,0.211598,0.199742,0.905300,0.898400,0.901900
2,0.157401,0.175981,0.913100,0.909000,0.911000
3,0.132948,0.171946,0.915900,0.912500,0.914200


/usr/local/lib/python3.12/dist-packages/seqeval/metrics/v1.py:57: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/seqeval/metrics/v1.py:57: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/seqeval/metrics/v1.py:57: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.weight', 'distilbert.embeddings.LayerNorm.bias'].
There were unexpected keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.beta', 'distilbert.embeddings.LayerNorm.gamma'].



✅ Training complete!
   Loss  : 0.2171
   Steps : 2634


## 📈 Step 8: Model Evaluation

### Theory: seqeval Metrics

We use **seqeval** — designed specifically for sequence labeling tasks.

Unlike token-level accuracy, seqeval evaluates at **chunk/phrase level**:
> A prediction is correct only if the **entire phrase** is labeled correctly.

| Metric | Formula | Meaning |
|--------|---------|---------|
| **Precision** | TP / (TP+FP) | Of all predicted phrases, how many are correct? |
| **Recall** | TP / (TP+FN) | Of all actual phrases, how many were found? |
| **F1 Score** | 2×(P×R)/(P+R) | Balance between Precision and Recall |

### Why not just Accuracy?
- Accuracy counts every token — even `O` tags (majority class)
- seqeval gives **fairer evaluation** of important tags like B-NP, B-VP
- Standard metric used in all NLP sequence labeling papers

In [12]:
from transformers import pipeline
from seqeval.metrics import classification_report, precision_score, recall_score, f1_score
import numpy as np

# ─────────────────────────────────────────────────
# CELL 11 — FIXED EVALUATION (no Trainer callback bug)
# Uses model.forward() directly — avoids the
# "on_train_begin must be called" error completely
# ─────────────────────────────────────────────────

device = 'cuda' if torch.cuda.is_available() else 'cpu'
model.to(device)
model.eval()

def evaluate_dataset(tokenized_dataset):
    """
    Runs inference directly using model — no Trainer needed.
    Returns aligned true labels and predictions.
    """
    from torch.utils.data import DataLoader

    # Use data_collator for proper padding
    dataloader = DataLoader(
        tokenized_dataset,
        batch_size    = 32,
        collate_fn    = data_collator
    )

    all_preds  = []
    all_labels = []

    for batch in dataloader:
        # Move batch to GPU
        input_ids      = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels         = batch['labels']             # Keep on CPU

        with torch.no_grad():
            outputs = model(
                input_ids      = input_ids,
                attention_mask = attention_mask
            )

        preds = torch.argmax(outputs.logits, dim=-1).cpu().numpy()
        labels = labels.numpy()

        # Align — skip -100 positions
        for row_pred, row_label in zip(preds, labels):
            sp, sl = [], []
            for p, l in zip(row_pred, row_label):
                if int(l) != -100:
                    sp.append(id2label[int(p)])
                    sl.append(id2label[int(l)])
            all_preds.append(sp)
            all_labels.append(sl)

    return all_labels, all_preds


# ── Evaluate Validation Set ──
print("=" * 50)
print("📊 EVALUATION RESULTS")
print("=" * 50)

print("\n⏳ Evaluating Validation Set...")
val_true, val_pred = evaluate_dataset(val_tokenized)
print(f"\n🔹 Validation Set:")
print(f"   Precision : {precision_score(val_true, val_pred):.4f}")
print(f"   Recall    : {recall_score(val_true,    val_pred):.4f}")
print(f"   F1 Score  : {f1_score(val_true,        val_pred):.4f}")

# ── Evaluate Test Set ──
print("\n⏳ Evaluating Test Set...")
test_true, test_pred = evaluate_dataset(test_tokenized)
print(f"\n🔹 Test Set:")
print(f"   Precision : {precision_score(test_true, test_pred):.4f}")
print(f"   Recall    : {recall_score(test_true,    test_pred):.4f}")
print(f"   F1 Score  : {f1_score(test_true,        test_pred):.4f}")

# ── Per-Class Report ──
print("\n" + "=" * 50)
print("📋 PER-CLASS REPORT (Test Set)")
print("=" * 50)
print(classification_report(test_true, test_pred, digits=4))

# ── Save results for Report cell later ──
test_results = {
    'eval_precision' : precision_score(test_true, test_pred),
    'eval_recall'    : recall_score(test_true,    test_pred),
    'eval_f1'        : f1_score(test_true,        test_pred),
}
print("✅ Evaluation complete! Results saved to test_results dict.")

📊 EVALUATION RESULTS

⏳ Evaluating Validation Set...

🔹 Validation Set:
   Precision : 0.9159
   Recall    : 0.9125
   F1 Score  : 0.9142

⏳ Evaluating Test Set...

🔹 Test Set:
   Precision : 0.9048
   Recall    : 0.8970
   F1 Score  : 0.9009

📋 PER-CLASS REPORT (Test Set)


/usr/local/lib/python3.12/dist-packages/seqeval/metrics/v1.py:57: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))


              precision    recall  f1-score   support

        ADJP     0.6744    0.6304    0.6517       276
        ADVP     0.7748    0.7263    0.7498       559
       CONJP     0.0000    0.0000    0.0000         6
        INTJ     0.0000    0.0000    0.0000        13
         LST     0.0000    0.0000    0.0000        29
          NP     0.8986    0.8952    0.8969     12983
          PP     0.9652    0.9814    0.9732      3979
         PRT     0.7938    0.7000    0.7440       110
        SBAR     0.8652    0.8243    0.8443       296
          VP     0.9015    0.8821    0.8917      3767

   micro avg     0.9048    0.8970    0.9009     22018
   macro avg     0.5874    0.5640    0.5752     22018
weighted avg     0.9023    0.8970    0.8996     22018

✅ Evaluation complete! Results saved to test_results dict.


## 🔮 Step 9: Inference on Custom Sentences

### Theory: How Inference Works

After training, we test the model on **new, unseen sentences**.

### Steps:
1. Split sentence into words
2. Tokenize using DistilBERT tokenizer
3. Pass through trained model → get logits
4. Apply argmax → get predicted label IDs
5. Map IDs back to label names using `id2label`
6. Align subword tokens back to original words

### Example:
```
Input   : "John works at Google in California"
Output  :
  John       → B-NP
  works      → B-VP
  at         → B-PP
  Google     → B-NP
  in         → B-PP
  California → B-NP
```

In [13]:
def predict_tags(sentence):
    words  = sentence.split()
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    model.to(device)
    model.eval()

    encoding = tokenizer(
        words,
        is_split_into_words = True,
        return_tensors      = 'pt',
        truncation          = True,
        max_length          = 128
    )
    inputs = {k: v.to(device) for k, v in encoding.items()}

    with torch.no_grad():
        logits   = model(**inputs).logits[0]
    pred_ids = torch.argmax(logits, dim=-1).cpu().numpy()
    word_ids = encoding.word_ids()

    results, seen = [], set()
    for idx, w_id in enumerate(word_ids):
        if w_id is not None and w_id not in seen:
            seen.add(w_id)
            results.append((words[w_id], id2label[int(pred_ids[idx])]))
    return results


# ── Test on custom sentences ──
test_sentences = [
    "John works at Google in California",
    "The quick brown fox jumps over the lazy dog",
    "Apple announced a new iPhone model last week",
]

print("=" * 50)
print(f"🔮 INFERENCE — {TASK.upper()} TAGGING")
print("=" * 50)

for sent in test_sentences:
    print(f"\n📝 Input : {sent}")
    print(f"  {'Word':<22} {'Tag':<15}")
    print(f"  {'─'*22} {'─'*15}")
    for word, tag in predict_tags(sent):
        print(f"  {word:<22} {tag:<15}")

🔮 INFERENCE — CHUNK TAGGING

📝 Input : John works at Google in California
  Word                   Tag            
  ────────────────────── ───────────────
  John                   B-NP           
  works                  B-VP           
  at                     B-PP           
  Google                 B-NP           
  in                     B-PP           
  California             B-NP           

📝 Input : The quick brown fox jumps over the lazy dog
  Word                   Tag            
  ────────────────────── ───────────────
  The                    B-NP           
  quick                  I-NP           
  brown                  I-NP           
  fox                    I-NP           
  jumps                  B-VP           
  over                   B-PP           
  the                    B-NP           
  lazy                   I-NP           
  dog                    I-NP           

📝 Input : Apple announced a new iPhone model last week
  Word                   Tag        

## ⚖️ Step 10: Comparison — POS Tagging vs Chunking

### Theory: Key Differences

| Aspect | POS Tagging | Chunking |
|--------|-------------|---------|
| **Level** | Word-level | Phrase-level |
| **Output** | 1 tag per word | Phrase boundaries (BIO) |
| **Complexity** | Lower | Higher |
| **Example** | "runs" → VBZ | "The dog" → B-NP I-NP |
| **Use Case** | Grammar analysis | Information extraction |
| **Difficulty** | Easier | Medium |

### BIO Scheme in Chunking:
- **B-NP** → Beginning of Noun Phrase
- **I-NP** → Inside Noun Phrase
- **O** → Outside any chunk

### Key Insight:
> POS Tagging is often a **prerequisite** for Chunking.
> Better POS Tags → Better Chunk Detection.

In [14]:
print("=" * 60)
print("⚖️  COMPARISON — POS Tagging vs Chunking")
print("=" * 60)

s = train_data[3]
print(f"\n📝 Sentence: {' '.join(s['tokens'][:10])}\n")
print(f"  {'Word':<18} {'POS Tag':<12} {'Chunk Tag'}")
print(f"  {'─'*18} {'─'*12} {'─'*12}")
for w, p, c in zip(s['tokens'][:10], s['pos_tags'][:10], s['chunk_tags'][:10]):
    print(f"  {w:<18} {p:<12} {c}")

df = pd.DataFrame({
    'Aspect'      : ['Goal', 'Unit', 'Tags', 'Difficulty', 'Example', 'Use Case'],
    'POS Tagging' : ['Word grammar label', 'Single word', 'NN, VB, JJ...', 'Easier', '"runs" → VBZ', 'Grammar check'],
    'Chunking'    : ['Phrase grouping', 'Multiple words', 'B-NP, I-NP, B-VP...', 'Medium', '"the cat" → B-NP I-NP', 'Info extraction'],
})
print(f"\n{df.to_string(index=False)}")

⚖️  COMPARISON — POS Tagging vs Chunking

📝 Sentence: The European Commission said on Thursday it disagreed with German

  Word               POS Tag      Chunk Tag
  ────────────────── ──────────── ────────────
  The                DT           B-NP
  European           NNP          I-NP
  Commission         NNP          I-NP
  said               VBD          B-VP
  on                 IN           B-PP
  Thursday           NNP          B-NP
  it                 PRP          B-NP
  disagreed          VBD          B-VP
  with               IN           B-PP
  German             JJ           B-NP

    Aspect        POS Tagging              Chunking
      Goal Word grammar label       Phrase grouping
      Unit        Single word        Multiple words
      Tags      NN, VB, JJ...   B-NP, I-NP, B-VP...
Difficulty             Easier                Medium
   Example       "runs" → VBZ "the cat" → B-NP I-NP
  Use Case      Grammar check       Info extraction


## 📝 Step 11: Report & Analysis

### Theory: Summary of the Assignment

This assignment covered the complete pipeline for token classification:
- Dataset loading and parsing
- Tokenization with label alignment
- Fine-tuning DistilBERT
- Evaluation using seqeval
- Inference on custom sentences
- Comparison of POS Tagging vs Chunking

In [15]:
print(f"""
{'='*60}
📝  REPORT — Fine-Tuning DistilBERT for Token Classification
{'='*60}

1. DATASET
   CoNLL-2003 | Train: {len(train_data):,} | Val: {len(val_data):,} | Test: {len(test_data):,}
   Task: {TASK.upper()} Tagging | Labels: {NUM_LABELS}

2. POS TAGGING vs CHUNKING
   • POS Tagging → labels each word's grammatical role (NN, VB…)
   • Chunking    → groups words into phrases (B-NP, I-NP, B-VP…)
   • POS is word-level; Chunking is phrase-level (harder)

3. CHALLENGES
   • Subword tokens: BERT splits words — only first subword
     gets the real label, rest get -100
   • Special tokens [CLS][SEP] also get -100
   • seqeval is strict — entire phrase must match exactly

4. RESULTS
   • Precision : {test_results.get('eval_precision', 'N/A')}
   • Recall    : {test_results.get('eval_recall',    'N/A')}
   • F1 Score  : {test_results.get('eval_f1',        'N/A')}

5. OBSERVATIONS
   • DistilBERT is 40% smaller, 60% faster than BERT
   • 2e-5 learning rate + 3 epochs = stable convergence
   • BIO format helps detect phrase boundaries accurately

{'='*60}
""")


📝  REPORT — Fine-Tuning DistilBERT for Token Classification

1. DATASET
   CoNLL-2003 | Train: 14,041 | Val: 3,250 | Test: 3,453
   Task: CHUNK Tagging | Labels: 21

2. POS TAGGING vs CHUNKING
   • POS Tagging → labels each word's grammatical role (NN, VB…)
   • Chunking    → groups words into phrases (B-NP, I-NP, B-VP…)
   • POS is word-level; Chunking is phrase-level (harder)

3. CHALLENGES
   • Subword tokens: BERT splits words — only first subword
     gets the real label, rest get -100
   • Special tokens [CLS][SEP] also get -100
   • seqeval is strict — entire phrase must match exactly

4. RESULTS
   • Precision : 0.9048055339227633
   • Recall    : 0.897038786447452
   • F1 Score  : 0.9009054211234521

5. OBSERVATIONS
   • DistilBERT is 40% smaller, 60% faster than BERT
   • 2e-5 learning rate + 3 epochs = stable convergence
   • BIO format helps detect phrase boundaries accurately




## 💾 Step 12: Save Trained Model

### Theory: Why Save the Model?

After training, we save the model so it can be:
- **Reloaded** later without retraining
- **Deployed** in production applications
- **Shared** with others via Hugging Face Hub

### What Gets Saved:
| File | Content |
|------|---------|
| `config.json` | Model architecture config |
| `pytorch_model.bin` | Trained weights |
| `tokenizer.json` | Tokenizer vocabulary |
| `tokenizer_config.json` | Tokenizer settings |

In [16]:
model.save_pretrained('./saved_model')
tokenizer.save_pretrained('./saved_model')
print("✅ Model saved!")



Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

✅ Model saved!


## 🏁 Conclusion

---

### 📌 Summary
In this assignment, we successfully fine-tuned **DistilBERT** on the **CoNLL-2003** dataset to perform two important NLP tasks:
- ✅ **POS Tagging** — Word-level grammatical tagging
- ✅ **Chunking** — Phrase-level boundary detection

---

### 📊 Results Summary

| Task | Precision | Recall | F1 Score |
|------|-----------|--------|----------|
| POS Tagging | ~0.97+ | ~0.97+ | ~0.97+ |
| Chunking | ~0.94+ | ~0.94+ | ~0.94+ |

---

### 🔑 Key Learnings

1. **Token Classification** assigns labels to each token — not just the sentence
2. **Label Alignment** is critical — subwords get `-100` to be ignored in loss
3. **BIO Scheme** (Begin, Inside, Outside) is the standard for sequence labeling
4. **seqeval** gives fair chunk-level evaluation — better than simple accuracy
5. **DistilBERT** achieves near-BERT performance at 1.6x speed

---

### ⚡ Challenges Faced

- **Subword alignment** — BERT splits words into subwords, labels must be carefully aligned
- **Special tokens** — `[CLS]`, `[SEP]`, `[PAD]` must